# QUBO + QAOA for VRP (Qiskit) - Step 1
Run the cells **top to bottom**.

What this notebook does: takes a small piece of a real routing dataset, turns it into a scoring puzzle (QUBO), solves it with QAOA on Qiskit's simulator, and compares the answer with classical methods (brute force, ILP, OR-tools, greedy).

Tip: set **Runtime -> Change runtime type -> CPU** (a GPU is not needed).

## 1. Install (pinned to the versions this was tested with)

In [ ]:
!pip install -q qiskit==2.5.2 qiskit-aer==0.17.2 scipy ortools

## 2. Create the four code files

In [ ]:
%%writefile vrp_qubo.py
"""
vrp_qubo.py  -  instances, QUBO encodings (arc / order) and QUBO -> Ising conversion.

Pure problem-formulation code (no quantum simulation here). The quantum part is in qaoa_qiskit.py.

Two encodings of a single-vehicle routing problem (depot = node 0):
  arc   : x_ij = 1  if the truck drives i -> j          (the paper's style; n*(n-1) qubits)
  order : x_ct = 1  if customer c is the t-th stop      ((n-1)^2 qubits)
"""
import itertools

import numpy as np
from qiskit.quantum_info import SparsePauliOp


# ---------------------------------------------------------------------------
# Instances
# ---------------------------------------------------------------------------
def load_instance(path):
    """Parse TSPLIB-like text files (Mavrovouniotis .evrp, CMT-style .vrp).
    VRP-REP XML files are not supported by this parser."""
    coords, demand, stations, depots, meta = {}, {}, set(), [], {}
    section = None
    with open(path) as f:
        for raw in f:
            line = raw.strip()
            if not line or line == "EOF":
                continue
            if line.upper().endswith("SECTION"):
                section = line.upper()
                continue
            if section is None:
                if ":" in line:
                    k, v = line.split(":", 1)
                    meta[k.strip()] = v.strip()
                continue
            parts = line.split()
            if section == "NODE_COORD_SECTION":
                coords[int(parts[0])] = (float(parts[1]), float(parts[2]))
            elif section == "DEMAND_SECTION":
                demand[int(parts[0])] = float(parts[1])
            elif section in ("STATIONS_COORD_SECTION", "STATION_COORD_SECTION"):
                stations.add(int(parts[0]))
            elif section == "DEPOT_SECTION":
                if int(parts[0]) != -1:
                    depots.append(int(parts[0]))
    depot = depots[0] if depots else min(coords)
    customers = sorted(i for i in coords if i != depot and i not in stations)
    return dict(depot=depot, customers=customers, coords=coords, demand=demand,
                stations=sorted(stations), meta=meta)


def dist_matrix(xy):
    xy = np.asarray(xy, float)
    return np.sqrt(((xy[:, None, :] - xy[None, :, :]) ** 2).sum(-1))


def subsample(inst, k, mode="nearest", seed=0):
    """Depot + k customers. mode: 'nearest' (k closest to depot) | 'first' | 'random'."""
    dxy = np.array(inst["coords"][inst["depot"]])
    cust = inst["customers"]
    if mode == "first":
        chosen = cust[:k]
    elif mode == "random":
        chosen = [int(c) for c in np.random.default_rng(seed).choice(cust, size=k, replace=False)]
    else:
        d = [np.linalg.norm(np.array(inst["coords"][c]) - dxy) for c in cust]
        chosen = [cust[i] for i in np.argsort(d)[:k]]
    xy = [inst["coords"][inst["depot"]]] + [inst["coords"][c] for c in chosen]
    return np.array(xy), [inst["depot"]] + chosen


def random_instance(k, seed=0, box=100.0):
    xy = np.random.default_rng(seed).uniform(0, box, size=(k + 1, 2))
    return xy, list(range(k + 1))


def route_cost(D, route):
    return float(sum(D[a, b] for a, b in zip(route[:-1], route[1:])))


# ---------------------------------------------------------------------------
# QUBO
# ---------------------------------------------------------------------------
class QUBO:
    """E(b) = const + sum_i lin[i] b_i + sum_{i<j} quad[i,j] b_i b_j   with b in {0,1}."""

    def __init__(self, n):
        self.n, self.const = n, 0.0
        self.lin = np.zeros(n)
        self.quad = {}

    def add_quad(self, i, j, v):
        if i == j:                                   # b^2 = b
            self.lin[i] += v
            return
        i, j = min(i, j), max(i, j)
        self.quad[(i, j)] = self.quad.get((i, j), 0.0) + v

    def add_exactly_one(self, idx, lam):
        """lam * (1 - sum b)^2 : zero only when exactly one of the switches is ON."""
        self.const += lam
        for i in idx:
            self.lin[i] -= lam
        for a in range(len(idx)):
            for b in range(a + 1, len(idx)):
                self.add_quad(idx[a], idx[b], 2.0 * lam)

    def mean_energy(self):
        """Average energy over all 2^n bitstrings (= what random guessing gives)."""
        return self.const + self.lin.sum() / 2 + sum(self.quad.values()) / 4

    def energy_of(self, b):
        """Energy of one bitstring given as int (bit k = qubit k)."""
        e = self.const + sum(self.lin[i] for i in range(self.n) if (b >> i) & 1)
        e += sum(v for (i, j), v in self.quad.items() if (b >> i) & 1 and (b >> j) & 1)
        return float(e)

    def all_energies(self):
        """Energy table of all 2^n bitstrings (classical, NumPy). Used ONLY to validate the
        penalty weights once; the quantum part never uses this table."""
        n = self.n
        idx = np.arange(1 << n, dtype=np.uint32)
        bits = [((idx >> k) & 1).astype(np.uint8) for k in range(n)]
        E = np.full(1 << n, self.const)
        for i in range(n):
            if self.lin[i]:
                E += self.lin[i] * bits[i]
        for (i, j), v in self.quad.items():
            E += v * (bits[i] & bits[j])
        return E

    def to_ising(self):
        """b = (1 - Z)/2   ->   const + sum h_i Z_i + sum J_ij Z_i Z_j"""
        c, h, J = self.const, np.zeros(self.n), {}
        for i in range(self.n):
            c += self.lin[i] / 2
            h[i] -= self.lin[i] / 2
        for (i, j), v in self.quad.items():
            c += v / 4
            h[i] -= v / 4
            h[j] -= v / 4
            J[(i, j)] = J.get((i, j), 0.0) + v / 4
        return c, h, J

    def ising_operator(self):
        """Returns (SparsePauliOp without the constant, constant). <H_qubo> = <op> + constant."""
        c, h, J = self.to_ising()
        terms = [("Z", [i], float(v)) for i, v in enumerate(h) if v != 0.0]
        terms += [("ZZ", [i, j], float(v)) for (i, j), v in J.items() if v != 0.0]
        return SparsePauliOp.from_sparse_list(terms, num_qubits=self.n), float(c)


# ---------------------------------------------------------------------------
# Encodings
# ---------------------------------------------------------------------------
class ArcEncoding:
    """x_ij = 1 if the truck drives i -> j (single vehicle, depot = node 0).

      cost    : sum d_ij x_ij
      visit   : every place has exactly one road leaving and one road arriving
                (this also covers the paper's depot constraint)
      subtour : lam * x_ij x_ji   (forbids 2-place loops, as in the paper)
    With <= 4 customers (5 places) that removes every possible broken loop. With more
    customers longer broken loops are possible; decode() rejects them as infeasible.
    """
    name = "arc"

    def __init__(self, D, lam=2.0):
        self.D = np.asarray(D, float)
        n = self.n_nodes = len(D)
        self.scale = self.D.max()
        Dn = self.D / self.scale
        self.var = {}
        for i in range(n):
            for j in range(n):
                if i != j:
                    self.var[(i, j)] = len(self.var)
        self.N, self.lam = len(self.var), lam
        q = self.qubo = QUBO(self.N)
        for (i, j), v in self.var.items():
            q.lin[v] += Dn[i, j]
        for i in range(n):
            q.add_exactly_one([self.var[i, j] for j in range(n) if j != i], lam)   # one road out
            q.add_exactly_one([self.var[j, i] for j in range(n) if j != i], lam)   # one road in
        for i in range(n):
            for j in range(i + 1, n):
                q.add_quad(self.var[i, j], self.var[j, i], lam)

    def encode(self, route):
        b = 0
        for a, c in zip(route[:-1], route[1:]):
            b |= 1 << self.var[(a, c)]
        return b

    def decode(self, b):
        n = self.n_nodes
        succ, indeg = {}, [0] * n
        for (i, j), v in self.var.items():
            if (b >> v) & 1:
                if i in succ:
                    return None
                succ[i] = j
                indeg[j] += 1
        if len(succ) != n or any(d != 1 for d in indeg):
            return None
        route, cur = [0], 0
        for _ in range(n):
            cur = succ[cur]
            route.append(cur)
        if route[-1] != 0 or len(set(route[:-1])) != n:        # must be ONE loop through every place
            return None
        return route


class OrderEncoding:
    """x_ct = 1 if customer c is the t-th stop (t = 0..m-1, m = number of customers).

      cost    : d(depot, 1st stop) + sum d(c, c') x_ct x_c'(t+1) + d(last stop, depot)
      penalty : every customer in exactly one position, every position holds exactly one customer
    Every valid answer is automatically one single loop, so no subtour term is needed.
    """
    name = "order"

    def __init__(self, D, lam=2.0):
        self.D = np.asarray(D, float)
        self.n_nodes = len(D)
        m = self.m = self.n_nodes - 1
        self.N, self.lam = m * m, lam
        self.scale = self.D.max()
        Dn = self.D / self.scale
        q = self.qubo = QUBO(self.N)
        for c in range(1, m + 1):
            q.lin[self._v(c, 0)] += Dn[0, c]
            q.lin[self._v(c, m - 1)] += Dn[c, 0]
        for t in range(m - 1):
            for c in range(1, m + 1):
                for c2 in range(1, m + 1):
                    if c != c2:
                        q.add_quad(self._v(c, t), self._v(c2, t + 1), Dn[c, c2])
        for c in range(1, m + 1):
            q.add_exactly_one([self._v(c, t) for t in range(m)], lam)
        for t in range(m):
            q.add_exactly_one([self._v(c, t) for c in range(1, m + 1)], lam)

    def _v(self, c, t):
        return (c - 1) * self.m + t

    def encode(self, route):
        b = 0
        for t, c in enumerate(route[1:-1]):
            b |= 1 << self._v(c, t)
        return b

    def decode(self, b):
        order = []
        for t in range(self.m):
            on = [c for c in range(1, self.m + 1) if (b >> self._v(c, t)) & 1]
            if len(on) != 1:
                return None
            order.append(on[0])
        if len(set(order)) != self.m:
            return None
        return [0, *order, 0]


ENCODINGS = {"arc": ArcEncoding, "order": OrderEncoding}


def build_validated(kind, D, optimal_routes, opt_cost, lams=(1.5, 2.0, 3.0, 5.0, 8.0),
                    max_check_qubits=22, verbose=True):
    """Choose the smallest penalty weight whose QUBO ground state(s) are exactly the optimal
    routes (checked against brute force). Classical check, done once before any quantum run."""
    for lam in lams:
        enc = ENCODINGS[kind](D, lam=lam)
        if enc.N > max_check_qubits:
            if verbose:
                print(f"  [{kind}] {enc.N} qubits: too many to validate by enumeration, using lam={lam}")
            return enc
        E = enc.qubo.all_energies()
        ground = np.flatnonzero(E <= E.min() + 1e-9)
        ok = abs(E.min() - opt_cost / enc.scale) < 1e-9 and all(
            (r := enc.decode(int(g))) is not None and abs(route_cost(D, r) - opt_cost) < 1e-6
            for g in ground)
        if verbose:
            print(f"  [{kind}] penalty weight {lam}: {'valid' if ok else 'ground state wrong, trying larger'}")
        if ok:
            return enc
    raise RuntimeError("No penalty weight produced a QUBO whose ground state is the optimal route.")


In [ ]:
%%writefile qaoa_qiskit.py
"""
qaoa_qiskit.py  -  QAOA implemented entirely with Qiskit.

  circuit      : |+>^n, then p x [ cost layer RZ/RZZ(gamma) , mixer RX(2 beta) ]
  objective    : <H_QUBO> from Aer's save_expectation_value on the parametrised circuit
                 (exact statevector simulation, circuit transpiled once, parameters re-bound each step)
  optimiser    : SciPy COBYLA (classical outer loop; this is what the RL agent replaces later)
  measurement  : Aer simulator, `shots` shots (like the paper's 1024/2048), plus exact
                 probabilities via Aer's save_probabilities for the "chance of the optimum" metric
"""
import time

import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.circuit import ParameterVector
from qiskit_aer import AerSimulator

from vrp_qubo import route_cost


def build_qaoa_circuit(qubo, p):
    """Parametrised QAOA circuit for the QUBO's Ising Hamiltonian."""
    const, h, J = qubo.to_ising()
    n = qubo.n
    gam, bet = ParameterVector("gamma", p), ParameterVector("beta", p)
    qc = QuantumCircuit(n)
    qc.h(range(n))
    for l in range(p):
        for i, hi in enumerate(h):
            if hi != 0.0:
                qc.rz(2 * gam[l] * float(hi), i)              # exp(-i gamma h Z)
        for (i, j), Jij in J.items():
            if Jij != 0.0:
                qc.rzz(2 * gam[l] * float(Jij), i, j)         # exp(-i gamma J Z Z)
        qc.rx(2 * bet[l], range(n))                           # mixer exp(-i beta X)
    return qc


class QAOA:
    def __init__(self, enc, p, seed=0):
        self.enc, self.p, self.n = enc, p, enc.N
        self.rng = np.random.default_rng(seed)
        self.seed = seed
        self.circuit = build_qaoa_circuit(enc.qubo, p)
        self.observable, self.const = enc.qubo.ising_operator()
        self.backend = AerSimulator(method="statevector")
        qc = self.circuit.copy()
        qc.save_expectation_value(self.observable, list(range(self.n)))
        self.exp_circuit = transpile(qc, self.backend, optimization_level=0)

    # -- objective ------------------------------------------------------------------
    def energy(self, params):
        """<H_QUBO> for the given (gamma_1..gamma_p, beta_1..beta_p)."""
        bound = self.exp_circuit.assign_parameters(np.asarray(params, float))
        ev = self.backend.run(bound).result().data()["expectation_value"]
        return float(ev) + self.const

    # -- classical outer loop ---------------------------------------------------------
    def optimise(self, restarts=3, maxiter=120):
        from scipy.optimize import minimize
        best = None
        for r in range(restarts):
            x0 = np.concatenate([self.rng.uniform(0.05, 1.0, self.p), self.rng.uniform(0, np.pi / 2, self.p)])
            trace = []

            def f(x):
                v = self.energy(x)
                trace.append(v)
                return v

            res = minimize(f, x0, method="COBYLA", options={"maxiter": maxiter, "rhobeg": 0.2})
            if best is None or res.fun < best["fun"]:
                best = dict(x=res.x, fun=float(res.fun), trace=trace)
        self.best = best
        return best

    # -- measurement ------------------------------------------------------------------
    def bound(self, params):
        return self.circuit.assign_parameters(np.asarray(params, float))

    def probabilities(self, params):
        """Exact outcome probabilities of the final state (Aer save_probabilities)."""
        qc = self.bound(params)
        qc.save_probabilities()
        res = self.backend.run(transpile(qc, self.backend, optimization_level=0)).result()
        return np.asarray(res.data()["probabilities"])

    def sample(self, params, shots=2048):
        """Measure `shots` times. Returns {bitstring_int: count}."""
        qc = self.bound(params)
        qc.measure_all()
        res = self.backend.run(transpile(qc, self.backend, optimization_level=0),
                               shots=shots, seed_simulator=self.seed).result()
        return {int(k.replace(" ", ""), 2): v for k, v in res.get_counts().items()}


def evaluate_counts(enc, counts, D, opt_cost):
    """Decode measured bitstrings; report feasibility, hit rate of the optimum, best cost."""
    tot = sum(counts.values())
    feas = hit = 0
    best = np.inf
    for b, c in counts.items():
        r = enc.decode(b)
        if r is None:
            continue
        cost = route_cost(D, r)
        feas += c
        best = min(best, cost)
        if cost <= opt_cost + 1e-6:
            hit += c
    return dict(feasible_frac=feas / tot, optimal_frac=hit / tot,
                best_cost=None if best == np.inf else float(best))


def run_qaoa(enc, D, opt_cost, optimal_routes, p, shots=2048, restarts=3, maxiter=120, seed=0):
    t0 = time.time()
    q = QAOA(enc, p, seed)
    best = q.optimise(restarts, maxiter)
    probs = q.probabilities(best["x"])
    ground = [enc.encode(r) for r in optimal_routes]
    p_opt = float(probs[ground].sum())
    uniform = len(ground) / (1 << enc.N)
    e_min, e_mean = opt_cost / enc.scale, enc.qubo.mean_energy()
    ev = evaluate_counts(enc, q.sample(best["x"], shots), D, opt_cost)
    gap = None if ev["best_cost"] is None else round(max(0.0, 100 * (ev["best_cost"] - opt_cost) / opt_cost), 4)
    return dict(p=p, params=best["x"].tolist(), expected_energy=best["fun"],
                energy_improvement=(e_mean - best["fun"]) / (e_mean - e_min),   # 0 = random, 1 = perfect
                prob_optimal=p_opt, boost_vs_uniform=p_opt / uniform,
                evaluations=len(best["trace"]), gap_pct=gap, seconds=time.time() - t0, **ev), best["trace"]


In [ ]:
%%writefile classical_baselines.py
"""
classical_baselines.py  -  the classical methods QAOA is compared against.

  brute_force : tries every order of customers. Exact. Also returns ALL optimal routes.
  ilp_mtz     : exact Integer Linear Program (arc variables + MTZ subtour elimination), solved with
                SciPy's HiGHS MILP solver. This is the "exact ILP" baseline used in the paper.
  ortools     : Google OR-Tools routing solver (guided local search heuristic).
  greedy      : nearest-neighbour heuristic (the paper's "Greedy Search").
"""
import itertools
import time

import numpy as np
from scipy.optimize import Bounds, LinearConstraint, milp

from vrp_qubo import route_cost


def brute_force(D, tol=1e-9):
    n = len(D)
    best, routes = np.inf, []
    for perm in itertools.permutations(range(1, n)):
        r = [0, *perm, 0]
        c = route_cost(D, r)
        if c < best - tol:
            best, routes = c, [r]
        elif abs(c - best) <= tol:
            routes.append(r)
    return best, routes


def greedy(D):
    n = len(D)
    route, left = [0], set(range(1, n))
    while left:
        nxt = min(left, key=lambda j: D[route[-1], j])
        route.append(nxt)
        left.remove(nxt)
    route.append(0)
    return route_cost(D, route), route


def ilp_mtz(D):
    """min sum d_ij x_ij ; one arc out / in per place ; MTZ: u_i - u_j + (n-1) x_ij <= n-2."""
    n = len(D)
    arcs = [(i, j) for i in range(n) for j in range(n) if i != j]
    na = len(arcs)
    nv = na + (n - 1)                                   # x_ij binary, then u_1..u_{n-1} continuous
    c = np.zeros(nv)
    for k, (i, j) in enumerate(arcs):
        c[k] = D[i, j]
    A, lo, hi = [], [], []
    for i in range(n):
        row = np.zeros(nv)
        for k, (a, b) in enumerate(arcs):
            if a == i:
                row[k] = 1
        A.append(row); lo.append(1); hi.append(1)       # one road out of i
        row = np.zeros(nv)
        for k, (a, b) in enumerate(arcs):
            if b == i:
                row[k] = 1
        A.append(row); lo.append(1); hi.append(1)       # one road into i
    for k, (i, j) in enumerate(arcs):
        if i != 0 and j != 0:
            row = np.zeros(nv)
            row[k] = n - 1
            row[na + i - 1] = 1
            row[na + j - 1] = -1
            A.append(row); lo.append(-np.inf); hi.append(n - 2)
    integrality = np.concatenate([np.ones(na), np.zeros(n - 1)])
    bounds = Bounds(np.concatenate([np.zeros(na), np.ones(n - 1)]),
                    np.concatenate([np.ones(na), np.full(n - 1, n - 1)]))
    res = milp(c, constraints=LinearConstraint(np.array(A), lo, hi), integrality=integrality, bounds=bounds)
    succ = {i: j for k, (i, j) in enumerate(arcs) if res.x[k] > 0.5}
    route, cur = [0], 0
    for _ in range(n):
        cur = succ[cur]
        route.append(cur)
    return route_cost(D, route), route


def ortools_solve(D, time_limit_s=1):
    from ortools.constraint_solver import pywrapcp, routing_enums_pb2
    n = len(D)
    scale = 1000
    Di = (np.asarray(D) * scale).round().astype(int)
    mgr = pywrapcp.RoutingIndexManager(n, 1, 0)
    routing = pywrapcp.RoutingModel(mgr)
    cb = routing.RegisterTransitCallback(lambda a, b: int(Di[mgr.IndexToNode(a), mgr.IndexToNode(b)]))
    routing.SetArcCostEvaluatorOfAllVehicles(cb)
    p = pywrapcp.DefaultRoutingSearchParameters()
    p.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    p.local_search_metaheuristic = routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
    p.time_limit.seconds = time_limit_s
    sol = routing.SolveWithParameters(p)
    idx, route = routing.Start(0), [0]
    while not routing.IsEnd(idx):
        idx = sol.Value(routing.NextVar(idx))
        route.append(mgr.IndexToNode(idx))
    return route_cost(D, route), route


def run_all(D):
    """Run every classical method, return {name: dict(cost, route, seconds)}."""
    out = {}
    methods = [("brute_force", lambda: (lambda r: (r[0], r[1][0]))(brute_force(D))),
               ("ilp_mtz", lambda: ilp_mtz(D)),
               ("greedy", lambda: greedy(D))]
    try:
        import ortools  # noqa: F401
        methods.insert(2, ("ortools", lambda: ortools_solve(D)))
    except ImportError:
        pass
    for name, fn in methods:
        t0 = time.time()
        cost, route = fn()
        out[name] = dict(cost=float(cost), route=route, seconds=time.time() - t0)
    return out


In [ ]:
%%writefile run_experiment.py
"""
run_experiment.py  -  QUBO + QAOA (Qiskit) vs classical methods on a small VRP instance.

Examples
  python run_experiment.py --file E-n29-k4-s7.evrp --customers 3 --kind both --p 1 2
  python run_experiment.py --customers 4 --kind order --p 1 2          # random instance
"""
import argparse
import json
import random

from classical_baselines import brute_force, run_all
from qaoa_qiskit import evaluate_counts, run_qaoa
from vrp_qubo import (build_validated, dist_matrix, load_instance, random_instance,
                      route_cost, subsample)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--file", help=".evrp / TSPLIB-like .vrp file. Omit for a random instance")
    ap.add_argument("--customers", type=int, default=3)
    ap.add_argument("--select", default="nearest", choices=["nearest", "first", "random"])
    ap.add_argument("--kind", default="both", choices=["arc", "order", "both"])
    ap.add_argument("--p", type=int, nargs="+", default=[1, 2], help="QAOA depths to try")
    ap.add_argument("--shots", type=int, default=2048)
    ap.add_argument("--restarts", type=int, default=3)
    ap.add_argument("--maxiter", type=int, default=120)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--out", default="results.json")
    a = ap.parse_args()

    xy, ids = subsample(load_instance(a.file), a.customers, a.select, a.seed) if a.file \
        else random_instance(a.customers, a.seed)
    D = dist_matrix(xy)
    print(f"Instance: depot + {a.customers} customers, original node ids {ids}")

    # ---- classical methods -------------------------------------------------------------
    classical = run_all(D)
    opt, opt_routes = brute_force(D)
    print("\nCLASSICAL METHODS")
    print(f"{'method':<14}{'cost':>10}{'gap %':>9}{'time (s)':>11}")
    for name, r in classical.items():
        print(f"{name:<14}{r['cost']:>10.3f}{100 * (r['cost'] - opt) / opt:>9.2f}{r['seconds']:>11.4f}")

    # ---- QAOA --------------------------------------------------------------------------
    kinds = ["arc", "order"] if a.kind == "both" else [a.kind]
    results = dict(instance_ids=ids, optimal_cost=opt, classical=classical, qaoa=[])
    for kind in kinds:
        print(f"\nQAOA ({kind} encoding)")
        enc = build_validated(kind, D, opt_routes, opt)
        print(f"  qubits: {enc.N}, penalty weight: {enc.lam}")

        rng = random.Random(a.seed)                        # no-quantum reference: random bitstrings
        rand_counts = {}
        for _ in range(a.shots):
            b = rng.getrandbits(enc.N)
            rand_counts[b] = rand_counts.get(b, 0) + 1
        rs = evaluate_counts(enc, rand_counts, D, opt)
        print(f"  random bitstrings : feasible {rs['feasible_frac']:.3f}  optimal {rs['optimal_frac']:.4f}"
              f"  best cost {rs['best_cost']}")

        for p in a.p:
            res, _ = run_qaoa(enc, D, opt, opt_routes, p, a.shots, a.restarts, a.maxiter, a.seed)
            res.update(kind=kind, qubits=enc.N, penalty=enc.lam, random_baseline=rs)
            results["qaoa"].append(res)
            print(f"  QAOA p={p}: P(optimal route)={res['prob_optimal']:.4f} "
                  f"(x{res['boost_vs_uniform']:.0f} vs random) | shots: feasible {res['feasible_frac']:.3f}, "
                  f"optimal {res['optimal_frac']:.3f}, best cost {res['best_cost']} (gap {res['gap_pct']}%) | "
                  f"energy improvement {res['energy_improvement']:.2f} | {res['evaluations']} evals, "
                  f"{res['seconds']:.1f}s")

    with open(a.out, "w") as f:
        json.dump(results, f, indent=2)
    print(f"\nSaved {a.out}")


if __name__ == "__main__":
    main()


## 3. Download the dataset
This is your E-VRP instance (29 places). We only use the warehouse plus a few customers.

In [ ]:
!wget -q -O E-n29-k4-s7.evrp https://raw.githubusercontent.com/Mavrovouniotis/e-cvrp_benchmark_instances/master/E-n29-k4-s7.evrp
!head -12 E-n29-k4-s7.evrp

## 4. Quick test: warehouse + 3 customers (about 10 seconds)
All classical methods should find the same best length. QAOA should find it too, but only on a small fraction of shots.

In [ ]:
!python run_experiment.py --file E-n29-k4-s7.evrp --customers 3 --kind both --p 1 2 --out r3.json 2>&1 | grep -v Warning

## 5. 4 customers, `order` layout (about 1-2 minutes)

In [ ]:
!python run_experiment.py --file E-n29-k4-s7.evrp --customers 4 --kind order --p 1 2 --out r4_order.json 2>&1 | grep -v Warning

## 6. 4 customers, `arc` layout (SLOW: 20 qubits, roughly 5-10 minutes)
Optional. Uses fewer restarts to save time.

In [ ]:
!python run_experiment.py --file E-n29-k4-s7.evrp --customers 4 --kind arc --p 1 2 --restarts 2 --maxiter 100 --out r4_arc.json 2>&1 | grep -v Warning

## 7. Summary table of all QAOA runs so far

In [ ]:
import json, glob
rows = []
for fn in sorted(glob.glob("r*.json")):
    r = json.load(open(fn))
    for q in r["qaoa"]:
        rows.append((fn, q["kind"], q["qubits"], q["p"], round(q["prob_optimal"], 5),
                     round(q["boost_vs_uniform"]), round(q["feasible_frac"], 3), q["best_cost"],
                     round(r["optimal_cost"], 3), q["gap_pct"], round(q["seconds"], 1)))
print(f"{'file':<14}{'kind':<7}{'qubits':>7}{'p':>3}{'P(best)':>10}{'x random':>10}{'valid%':>8}{'QAOA best':>11}{'true best':>11}{'gap%':>7}{'sec':>7}")
for r in rows:
    print(f"{r[0]:<14}{r[1]:<7}{r[2]:>7}{r[3]:>3}{r[4]:>10}{r[5]:>10}{r[6]:>8}{r[7]:>11.3f}{r[8]:>11.3f}{r[9]:>7}{r[10]:>7}")

## 8. Optional: use your own file (e.g. a CMT instance)
Upload a **text-format** file (`.vrp` / `.evrp`, TSPLIB style). The VRP-REP CMT download is XML and will not load; convert it or ask for an XML reader.

In [ ]:
from google.colab import files
up = files.upload()
name = list(up.keys())[0]
!python run_experiment.py --file "{name}" --customers 4 --kind order --p 1 2 --out r_custom.json 2>&1 | grep -v Warning

## How to read the results
- **P(best)**: chance that ONE measurement gives the best route. Classical methods get it 100% of the time.
- **x random**: how many times better that is than random guessing.
- **valid%**: share of measurements that are real routes at all.
- **gap%**: how much longer QAOA's best route is than the true best. 0 means it found the best route.

Do not go above 4 customers in this notebook: 5 customers needs 25 qubits (several GB of RAM) and will likely crash free Colab.